# AWS RL Agent — SFT with LoRA on Qwen2.5-Coder-3B

**Pipeline**: HF SFT dataset → Optuna-tuned LoRA SFT → HF Hub adapter → GRPO (next phase)

| | |
|---|---|
| **Base model** | `Qwen/Qwen2.5-Coder-3B-Instruct` (4-bit via Unsloth) |
| **Dataset** | [`Sizzing/aws-rl-sft`](https://huggingface.co/datasets/Sizzing/aws-rl-sft) — 1500 train / 150 val |
| **Target GPU** | Kaggle dual-T4 (single T4 is enough with 4-bit) |
| **Expected runtime** | ~45 min end-to-end |
| **Logging** | matplotlib PNGs in `OUT_DIR/plots/` |

## What this notebook does

1. **Pre-SFT baseline eval** — zero-shot metrics on the base model (the "before")
2. **Optuna search** — 6 trials on a 500-row subset to pick best LoRA hparams
3. **Final SFT** — full dataset with winning hparams, checkpointed to disk
4. **Post-SFT eval** — same prompts, measure the delta
5. **Push adapter** — 60MB LoRA adapter to HF Hub (not the full 3B model)

## Why this stack

- **Unsloth** — ~2× training speed and half the VRAM via fused kernels (makes 3B fit on a single T4)
- **Optuna** — systematic hparam search instead of one-shot guessing; produces the parallel-coord plot judges love
- **TRL `SFTTrainer`** — handles the `messages` column and chat template automatically
- **matplotlib** — every training/Optuna/eval figure rendered locally and saved as PNG to `OUT_DIR/plots/` for download

## Expected deltas (from the pre-training eval)

| Metric | Before | Target after |
|---|--:|--:|
| `format%` | 85% | 100% |
| `exact-match%` | 41% | 75%+ |
| `operation%` | 63% | 90%+ |

## Checkpoint / resume

Final training saves every 50 steps to `/kaggle/working/final_sft/`. If the session dies, rerunning the final-training cell auto-detects the latest `checkpoint-*/` and resumes from it.

## Reading guide

The notebook is structured **before-vs-after**: every metric you see in the post-SFT section (§12) has a baseline counterpart from §7. The Optuna plots in §10 explain *why* the final hparams were picked; the training curves in §11a verify the run was healthy. If you only have time for one screen, it's the comparison bar chart in §12.

## 1. Install dependencies

Unsloth pulls in `trl`, `peft`, `accelerate`, `bitsandbytes`, and `transformers` at compatible versions. This takes ~3-4 min on first run.

The cell uses a `%%capture` magic to keep the noisy install logs out of the notebook. After it runs, the runtime has:

- **`unsloth`** — the fast fine-tuning toolkit (fused kernels for 4-bit Qwen2)
- **`transformers>=4.50,<5.0`** — pinned away from 5.x because Unsloth still passes `tokenizer=` to the trainer (renamed in v5; we patch this with a shim later if needed)
- **`trl<0.12.0`** — `SFTTrainer` API used here; v0.12 reshuffled the dataset-prep path
- **`peft`, `accelerate`, `bitsandbytes`** — LoRA, distributed-training utilities, 4-bit quantisation
- **`optuna` + `optuna-integration`** — the hyperparameter search itself plus its visualisation hooks
- **`datasets`, `huggingface_hub`** — pull the SFT dataset and (later) push the adapter

`IS_KAGGLE` / `IS_COLAB` are toggled at the top — they decide which Unsloth extras package to use and where the output directory lives. The current notebook is set up for Colab.

Because of `%%capture`, you won't see pip's progress bars; the cell is silent on success. If something fails, drop the magic and re-run to surface the error.

In [ ]:
%%capture
# Unsloth ships platform-specific extras pinning xformers / triton / torch
# to versions that match the runtime. Pick the right one based on host.
import os as _os, sys as _sys
IS_KAGGLE = False
IS_COLAB = True

!pip install -q --upgrade pip

# if IS_KAGGLE:
#     !pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
# elif IS_COLAB:
#     !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# else:
#     !pip install -q unsloth

!pip install -q unsloth

!pip install -q --force-reinstall --no-deps "transformers>=4.50,<5.0"

!pip install -q --upgrade "trl<0.12.0" peft accelerate datasets huggingface_hub bitsandbytes
!pip install -q --upgrade optuna optuna-integration

## 2. Runtime sanity check

Confirm the GPU, pick the right output directory (Kaggle vs Colab vs local), and fail loudly if there's no GPU.

This cell pins three things you'll need later:

1. **`OUT_DIR`** — host-specific destination for plots, checkpoints, the adapter, and metric JSONs. `/kaggle/working` on Kaggle, `/content/out` on Colab, `./out` otherwise.
2. **`PYTORCH_ALLOC_CONF=expandable_segments:True`** — reduces VRAM fragmentation when models load and free repeatedly (matters during the 6 Optuna trials, where each trial reloads a fresh base model).
3. **`USE_FP16` / `USE_BF16`** — Tesla T4 (Turing arch, CC 7.5) has no native bf16 support, so the cell forces fp16 there. Other GPUs (A100, H100, T4 successors) prefer bf16. Being explicit avoids silent precision conversions inside Unsloth.

### What the output tells you

```
Environment  : Colab
Output dir   : /content/out
GPU          : Tesla T4
VRAM         : 15.6 GB
GPU count    : 1
Torch        : 2.10.0+cu128
CUDA         : 12.8
Precision    : fp16
```

The `assert torch.cuda.is_available()` line halts the notebook with a clear error if the runtime is CPU-only — fail loud, fail early. 15.6 GB of VRAM is just enough to fit the 4-bit Qwen2.5-Coder-3B plus the LoRA adapter and a per-device batch of 2; if you see a smaller GPU, drop `per_device_train_batch_size` in the next cell.

In [ ]:
import os, sys, json, time, gc, re
from pathlib import Path
import torch


if IS_KAGGLE:
    OUT_DIR = Path('/kaggle/working')
elif IS_COLAB:
    OUT_DIR = Path('/content/out')
else:
    OUT_DIR = Path('./out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

assert torch.cuda.is_available(), 'No GPU detected — enable GPU in runtime settings.'
gpu = torch.cuda.get_device_properties(0)

print(f'Environment  : {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')
print(f'Output dir   : {OUT_DIR}')
print(f'GPU          : {gpu.name}')
print(f'VRAM         : {gpu.total_memory / 1e9:.1f} GB')
print(f'GPU count    : {torch.cuda.device_count()}')
print(f'Torch        : {torch.__version__}')
print(f'CUDA         : {torch.version.cuda}')

# T4 (Turing) does not support bf16. We use fp16 throughout. Unsloth also
# handles this internally, but being explicit avoids silent conversions.
IS_T4 = 'T4' in gpu.name
USE_FP16 = IS_T4
USE_BF16 = not IS_T4
print(f'Precision    : {"fp16" if USE_FP16 else "bf16"}')

Environment  : Colab
Output dir   : /content/out
GPU          : Tesla T4
VRAM         : 15.6 GB
GPU count    : 1
Torch        : 2.10.0+cu128
CUDA         : 12.8
Precision    : fp16


## 3. Config

Every knob lives in one dict. Fixed hyperparameters (batch size, epochs, target modules) stay here; Optuna tunes the ones below marked `(TUNED)`.

The dict is split into five sections — keeping them in one place makes it easy to fork the notebook for a different model or dataset without hunting through cells.

| Section | Keys | Notes |
|---|---|---|
| **Model** | `base_model`, `max_seq_length`, `load_in_4bit` | Unsloth's pre-quantised 4-bit checkpoint loads ~4× faster than letting bitsandbytes quantise on the fly. `max_seq_length=512` covers the dataset's p99 (≈453 tokens) with a small safety margin. |
| **Dataset** | `dataset_repo` | HF Hub repo for the SFT dataset. |
| **Fixed SFT hparams** | `per_device_train_batch_size=2`, `gradient_accumulation_steps=8`, `num_train_epochs=2`, `optim='adamw_8bit'`, `lr_scheduler_type='cosine'`, `weight_decay=0.0`, `seed=42`, `lora_target_modules=[q_proj, k_proj, v_proj, o_proj]` | Effective batch = 2 × 8 = 16 sequences. `adamw_8bit` saves ~25% VRAM vs. fp32 Adam. LoRA only attaches to the four attention projections — MLP projections are skipped to keep the adapter small (~15 MB). |
| **Optuna search (TUNED)** | `n_trials=6`, `trial_max_train_rows=500`, `trial_epochs=1` | A short, *ranking-only* search: 6 trials × 1 epoch × 500 rows ≈ 25 min on a T4. The aim is to rank hparams, not converge them. |
| **Checkpointing** | `save_steps=50`, `save_total_limit=3`, `eval_steps=50`, `logging_steps=10` | Every 50 steps; only the last 3 checkpoints survive on disk to keep the working directory under Kaggle's quota. |
| **Output** | `adapter_repo`, `adapter_private` | Where the LoRA adapter is pushed at the end. Private by default so you can iterate before sharing. |

The trailing `CONFIG` line returns the dict so Jupyter renders it inline — a quick visual confirmation that nothing got mistyped before training kicks off.

In [ ]:
CONFIG = {
    # --- Model ---
    # Unsloth's 4-bit pre-quantized version loads ~4× faster than bnb-on-the-fly
    'base_model': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
    'max_seq_length': 512,          # our dataset p99 is 453; 512 covers everything
    'load_in_4bit': True,

    # --- Dataset ---
    'dataset_repo': 'Sizzing/aws-rl-sft',

    # --- Fixed SFT hyperparameters ---
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,    # effective batch = 16
    'num_train_epochs': 2,               # full final run
    'optim': 'adamw_8bit',               # Unsloth-compatible, saves VRAM
    'max_grad_norm': 1.0,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.0,                 # LoRA is self-regularizing
    'seed': 42,
    'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],

    # --- Optuna search (TUNED) ---
    'n_trials': 6,
    'trial_max_train_rows': 500,         # subset for fast trials
    'trial_epochs': 1,                   # one pass is enough to rank hparams

    # --- Checkpointing ---
    'save_steps': 50,
    'save_total_limit': 3,               # keep last 3 checkpoints
    'logging_steps': 10,
    'eval_steps': 50,

    # --- Output ---
    'adapter_repo': 'Sizzing/aws-rl-sft-qwen25coder3b-adapter',
    'adapter_private': True,
}

CONFIG

## 3a. Plotting helpers (matplotlib)

Defines the plot functions and `PLOTS_DIR` used throughout the notebook. Every figure produced later (training curves, Optuna views, eval comparison) is rendered inline **and** saved to `OUT_DIR/plots/` as a PNG so it can be downloaded.

The cell registers seven plot helpers and a shared colour palette so all artefacts have one visual language (matches `compare_base_vs_sft.ipynb`):

| Helper | Used in | What it shows |
|---|---|---|
| `plot_training_curves(log_history, …)` | §11a (final SFT) | 2×2 grid: train loss, eval loss, LR schedule, gradient norm |
| `plot_optuna_history(study)` | §10 | Per-trial objective + best-so-far line — answers "did later trials improve?" |
| `plot_optuna_param_importances(study)` | §10 | fANOVA-style horizontal bar of relative importance — which knob mattered |
| `plot_optuna_parallel_coordinates(study)` | §10 | Polylines connecting hparams; viridis colour = objective (lower = better) |
| `plot_optuna_slice(study)` | §10 | One scatter per param against the objective, coloured by trial number |
| `plot_optuna_trial_loss_curves(study)` | §10 | Overlay every trial's training/eval curves — relies on `log_history` stashed via `trial.set_user_attr` inside `objective()` |
| `plot_eval_comparison(baseline, posttrain)` | §12 | Grouped bars (before vs after) + delta-pp bar (green = improvement, red = regression) |

The internal helpers `_save()` and `_split_log_history()` are private utilities. `_save()` writes PNGs at 150 DPI with a white background so the images display cleanly on dark slides too. `_split_log_history()` separates the trainer's mixed list of train/eval/summary records — train rows have `loss`+`step`, eval rows have `eval_loss`+`step`, the final summary row (no step) is dropped.

### What the output tells you

A single confirmation line:

```
Plot helpers ready. Figures will be saved to: <PLOTS_DIR>
```

No figure renders here — the cell only registers helpers.

In [ ]:
# ─── Plotting helpers ──────────────────────────────────────────────────────
# All training-time charts (loss/eval/lr/grad-norm + Optuna views + eval bars)
# are matplotlib figures, displayed inline and saved as PNGs to OUT_DIR/plots/.
# Helpers:
#   plot_training_curves(...)              — train/eval loss + lr + grad_norm
#   plot_optuna_history(...)               — best-so-far + per-trial scatter
#   plot_optuna_param_importances(...)     — fANOVA-style horizontal bar
#   plot_optuna_parallel_coordinates(...)  — per-trial polylines
#   plot_optuna_slice(...)                 — each param vs objective
#   plot_optuna_trial_loss_curves(...)     — overlay every trial's loss curves
#   plot_eval_comparison(...)              — baseline vs post-SFT bars + delta
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

PLOTS_DIR = OUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Colour palette mirrors compare_base_vs_sft.ipynb so all artefacts share one
# visual language.
COLOR_BASELINE = '#4C72B0'
COLOR_TRAINED  = '#DD8452'
COLOR_POS      = '#55A868'
COLOR_NEG      = '#C44E52'
COLOR_TRAIN    = '#4C72B0'
COLOR_EVAL     = '#DD8452'
COLOR_GRAD     = '#8172B2'
COLOR_LR       = '#937860'

plt.rcParams.update({
    'figure.dpi':        120,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'axes.labelsize':    10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'legend.framealpha': 0.9,
    'legend.fontsize':   9,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
})


def _save(fig, name):
    """Save a figure to PLOTS_DIR/<name>.png at dpi=150 with white background."""
    path = PLOTS_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'  saved -> {path}')
    return path


def _split_log_history(log_history):
    """Split trainer.state.log_history into (train_records, eval_records).

    Train rows have 'loss' + 'step'; eval rows have 'eval_loss' + 'step'. The
    final summary row (no step) is dropped.
    """
    train, eval_ = [], []
    for r in log_history:
        if 'loss' in r and 'step' in r:
            train.append(r)
        if 'eval_loss' in r and 'step' in r:
            eval_.append(r)
    return train, eval_


def plot_training_curves(log_history, *, title, save_name):
    """Render train/eval loss + LR + grad-norm in a 2x2 grid."""
    train, eval_ = _split_log_history(log_history)
    if not train:
        print(f'  (no training records to plot for {save_name})')
        return None

    t_step = np.array([r['step'] for r in train])
    t_loss = np.array([r['loss'] for r in train])
    t_lr   = np.array([r.get('learning_rate', np.nan) for r in train])
    t_gn   = np.array([r.get('grad_norm', np.nan)     for r in train])

    fig = plt.figure(figsize=(14, 9))
    gs  = GridSpec(2, 2, figure=fig, hspace=0.36, wspace=0.28)
    ax_loss = fig.add_subplot(gs[0, 0])
    ax_eval = fig.add_subplot(gs[0, 1])
    ax_lr   = fig.add_subplot(gs[1, 0])
    ax_gn   = fig.add_subplot(gs[1, 1])

    ax_loss.plot(t_step, t_loss, color=COLOR_TRAIN, lw=1.6, label='train_loss')
    ax_loss.set(title='Training loss vs step', xlabel='Step', ylabel='Cross-entropy loss')
    ax_loss.legend(loc='upper right')

    if eval_:
        e_step = np.array([r['step']      for r in eval_])
        e_loss = np.array([r['eval_loss'] for r in eval_])
        ax_eval.plot(e_step, e_loss, color=COLOR_EVAL, lw=1.6,
                     marker='o', markersize=5, label='eval_loss')
        ax_eval.set(title='Eval loss vs step', xlabel='Step', ylabel='Eval cross-entropy loss')
        ax_eval.legend(loc='upper right')
    else:
        ax_eval.text(0.5, 0.5, 'No eval records', ha='center', va='center',
                     transform=ax_eval.transAxes, color='#888')
        ax_eval.set(title='Eval loss vs step', xlabel='Step', ylabel='Eval cross-entropy loss')

    if not np.all(np.isnan(t_lr)):
        ax_lr.plot(t_step, t_lr, color=COLOR_LR, lw=1.6, label='learning_rate')
        ax_lr.set(title='Learning rate schedule', xlabel='Step', ylabel='Learning rate')
        ax_lr.legend(loc='upper right')
    else:
        ax_lr.text(0.5, 0.5, 'No LR records', ha='center', va='center',
                   transform=ax_lr.transAxes, color='#888')
        ax_lr.set(title='Learning rate schedule', xlabel='Step', ylabel='Learning rate')

    if not np.all(np.isnan(t_gn)):
        ax_gn.plot(t_step, t_gn, color=COLOR_GRAD, lw=1.6, label='grad_norm')
        ax_gn.set(title='Gradient norm vs step', xlabel='Step', ylabel='||grad||₂')
        ax_gn.legend(loc='upper right')
    else:
        ax_gn.text(0.5, 0.5, 'No grad-norm records', ha='center', va='center',
                   transform=ax_gn.transAxes, color='#888')
        ax_gn.set(title='Gradient norm vs step', xlabel='Step', ylabel='||grad||₂')

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    _save(fig, save_name)
    plt.show()
    return fig


def plot_optuna_history(study, *, save_name='optuna_history'):
    """Best-so-far line + per-trial scatter (matplotlib version of plot_optimization_history)."""
    trials = [t for t in study.trials if t.value is not None]
    if not trials:
        print('  (no completed trials to plot)'); return None
    nums = np.array([t.number for t in trials])
    vals = np.array([t.value  for t in trials])
    best_so_far = np.minimum.accumulate(vals)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(nums, vals, s=55, color=COLOR_TRAIN, edgecolors='white',
               linewidths=0.6, label='Trial value', zorder=3)
    ax.plot(nums, best_so_far, color=COLOR_NEG, lw=1.8, marker='s',
            markersize=6, label='Best so far', zorder=2)
    ax.set(title='Optuna optimization history',
           xlabel='Trial number',
           ylabel=f'Objective (eval_loss, {study.direction.name.lower()})')
    ax.set_xticks(nums)
    ax.legend(loc='upper right')
    plt.tight_layout()
    _save(fig, save_name)
    plt.show()
    return fig


def plot_optuna_param_importances(study, *, save_name='optuna_importances'):
    """fANOVA-style horizontal bar of optuna.importance.get_param_importances."""
    try:
        import optuna
        importances = optuna.importance.get_param_importances(study)
    except Exception as e:
        print(f'  (param importances unavailable: {e})')
        return None
    params = list(importances.keys())
    vals   = list(importances.values())
    order  = np.argsort(vals)[::-1]
    params = [params[i] for i in order]
    vals   = [vals[i]   for i in order]

    fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(params) + 1.5)))
    bars = ax.barh(params[::-1], vals[::-1], color=COLOR_TRAIN,
                   edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=9, fontweight='bold')
    ax.set(title='Hyperparameter importance (fANOVA-style)',
           xlabel='Relative importance', ylabel='Hyperparameter')
    if vals:
        ax.set_xlim(0, max(vals) * 1.15)
    plt.tight_layout()
    _save(fig, save_name)
    plt.show()
    return fig


def plot_optuna_parallel_coordinates(study, *, save_name='optuna_parallel'):
    """Per-trial polyline plot. Color = objective value (lower = better)."""
    trials = [t for t in study.trials if t.value is not None]
    if not trials:
        print('  (no completed trials to plot)'); return None

    param_names = []
    for t in trials:
        for k in t.params:
            if k not in param_names:
                param_names.append(k)

    n_trials = len(trials)
    n_dims   = len(param_names) + 1
    matrix   = np.zeros((n_trials, n_dims))
    tick_labels = {}

    for d, p in enumerate(param_names):
        col = [t.params.get(p) for t in trials]
        if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in col):
            matrix[:, d] = col
        else:
            uniques = sorted({str(v) for v in col})
            mp = {u: i for i, u in enumerate(uniques)}
            matrix[:, d] = [mp[str(v)] for v in col]
            tick_labels[d] = uniques
    matrix[:, -1] = [t.value for t in trials]
    obj_label = 'objective (eval_loss)'

    norm = np.zeros_like(matrix)
    for d in range(n_dims):
        col = matrix[:, d]
        lo, hi = col.min(), col.max()
        norm[:, d] = (col - lo) / (hi - lo) if hi > lo else 0.5

    obj_vals = matrix[:, -1]
    cmap     = plt.cm.viridis_r
    spread   = np.ptp(obj_vals) if np.ptp(obj_vals) else 1.0
    obj_norm = (obj_vals - obj_vals.min()) / spread

    fig, ax = plt.subplots(figsize=(12, 5.5))
    xs = np.arange(n_dims)
    for i in range(n_trials):
        ax.plot(xs, norm[i], color=cmap(obj_norm[i]), alpha=0.85, lw=1.5)

    for d in range(n_dims):
        ax.axvline(d, color='#bbb', lw=0.8, zorder=1)

    ax.set_xticks(xs)
    ax.set_xticklabels(param_names + [obj_label], rotation=20, ha='right')
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_yticklabels(['min', 'mid', 'max'])
    ax.set(title='Optuna parallel coordinates (color = objective; lower is better)',
           xlabel='Hyperparameter', ylabel='Normalised value')
    for d, labels in tick_labels.items():
        text = f'{param_names[d]}: ' + ', '.join(f'{i}={l}' for i, l in enumerate(labels))
        ax.annotate(text, xy=(d, -0.18), xycoords=('data', 'axes fraction'),
                    ha='center', fontsize=7, color='#555')

    sm = plt.cm.ScalarMappable(cmap=cmap,
                               norm=plt.Normalize(vmin=obj_vals.min(), vmax=obj_vals.max()))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(obj_label, fontsize=9)
    plt.tight_layout()
    _save(fig, save_name)
    plt.show()
    return fig


def plot_optuna_slice(study, *, save_name='optuna_slice'):
    """Per-parameter scatter: param value vs objective, colored by trial number."""
    trials = [t for t in study.trials if t.value is not None]
    if not trials:
        print('  (no completed trials)'); return None
    param_names = []
    for t in trials:
        for k in t.params:
            if k not in param_names:
                param_names.append(k)
    n = len(param_names)
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5.0 * cols, 4.0 * rows),
                             squeeze=False, constrained_layout=True)
    sc = None
    for i, p in enumerate(param_names):
        ax = axes[i // cols][i % cols]
        xs   = [t.params.get(p) for t in trials]
        ys   = [t.value          for t in trials]
        nums = [t.number         for t in trials]
        if not all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in xs):
            uniques = sorted({str(v) for v in xs})
            mp = {u: i for i, u in enumerate(uniques)}
            xs_plot = [mp[str(v)] for v in xs]
            ax.set_xticks(list(range(len(uniques))))
            ax.set_xticklabels(uniques)
        else:
            xs_plot = xs
        sc = ax.scatter(xs_plot, ys, c=nums, cmap='viridis', s=60,
                        edgecolors='white', linewidths=0.6)
        ax.set(title=p, xlabel=p, ylabel='eval_loss')
    for j in range(n, rows * cols):
        axes[j // cols][j % cols].axis('off')
    if sc is not None:
        cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), pad=0.02, fraction=0.04)
        cbar.set_label('Trial number', fontsize=9)
    fig.suptitle('Optuna slice plot (objective vs each hyperparameter)',
                 fontsize=14, fontweight='bold')
    _save(fig, save_name)
    plt.show()
    return fig


def plot_optuna_trial_loss_curves(study, *, save_name='optuna_trial_curves'):
    """Overlay every trial's training-loss + eval-loss curves.

    Reads `log_history` from each trial's user_attrs (set inside objective()).
    """
    trials_with_history = [t for t in study.trials
                           if t.value is not None
                           and 'log_history' in t.user_attrs]
    if not trials_with_history:
        print('  (no per-trial log_history captured — skipping trial curves)')
        return None

    cmap = plt.cm.viridis
    n    = len(trials_with_history)

    fig, (ax_t, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    for i, t in enumerate(trials_with_history):
        log = t.user_attrs['log_history']
        train, eval_ = _split_log_history(log)
        color = cmap(i / max(1, n - 1))
        if train:
            ax_t.plot([r['step'] for r in train], [r['loss'] for r in train],
                      color=color, lw=1.4, alpha=0.9,
                      label=f'trial {t.number} (loss={t.value:.3f})')
        if eval_:
            ax_e.plot([r['step'] for r in eval_], [r['eval_loss'] for r in eval_],
                      color=color, lw=1.4, marker='o', markersize=4, alpha=0.9,
                      label=f'trial {t.number} (loss={t.value:.3f})')

    ax_t.set(title='Per-trial training loss', xlabel='Step', ylabel='Cross-entropy loss')
    ax_e.set(title='Per-trial eval loss',     xlabel='Step', ylabel='Eval cross-entropy loss')
    ax_t.legend(loc='upper right', fontsize=7)
    ax_e.legend(loc='upper right', fontsize=7)
    fig.suptitle('Optuna — per-trial training & eval curves', fontweight='bold', y=1.02)
    plt.tight_layout()
    _save(fig, save_name)
    plt.show()
    return fig


def plot_eval_comparison(baseline, posttrain, *, save_name='eval_metrics_comparison'):
    """Grouped bar (baseline vs post-SFT) + delta bar for the 4 eval metrics."""
    keys   = ['format_pct', 'exact_pct', 'service_pct', 'operation_pct']
    labels = ['Format', 'Exact', 'Service', 'Operation']
    base_v = [100 * baseline[k]  for k in keys]
    post_v = [100 * posttrain[k] for k in keys]
    delta  = [p - b for b, p in zip(base_v, post_v)]

    fig = plt.figure(figsize=(14, 5.5))
    gs  = GridSpec(1, 2, figure=fig, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    x, w = np.arange(len(labels)), 0.35
    bars1 = ax1.bar(x - w/2, base_v, w, color=COLOR_BASELINE, label='Pre-SFT (baseline)',
                    edgecolor='white', linewidth=0.6)
    bars2 = ax1.bar(x + w/2, post_v, w, color=COLOR_TRAINED,  label='Post-SFT',
                    edgecolor='white', linewidth=0.6)
    for bars in (bars1, bars2):
        for bar in bars:
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.1f}%', ha='center', va='bottom',
                     fontsize=9, fontweight='bold')
    ax1.set(title='Eval metrics — Baseline vs Post-SFT', xlabel='Metric',
            ylabel='Score (%)', ylim=(0, 118))
    ax1.set_xticks(x); ax1.set_xticklabels(labels)
    ax1.legend(loc='lower right')

    colors = [COLOR_POS if d >= 0 else COLOR_NEG for d in delta]
    bars = ax2.bar(x, delta, 0.5, color=colors, edgecolor='white', linewidth=0.6)
    for bar, d in zip(bars, delta):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.6 if d >= 0 else -1.6),
                 f'{d:+.1f}pt', ha='center',
                 va='bottom' if d >= 0 else 'top',
                 fontsize=9, fontweight='bold')
    ax2.axhline(0, color='#333', lw=0.9)
    ax2.set(title='Delta (Post − Pre, percentage points)', xlabel='Metric', ylabel='Δ pp')
    ax2.set_xticks(x); ax2.set_xticklabels(labels)
    ax2.legend(handles=[mpatches.Patch(color=COLOR_POS, label='Improvement'),
                        mpatches.Patch(color=COLOR_NEG, label='Regression')])
    plt.tight_layout()
    _save(fig, save_name)
    plt.show()
    return fig


print(f'Plot helpers ready. Figures will be saved to: {PLOTS_DIR}')

## 4. Authenticate

`HF_TOKEN` must be set so we can pull the dataset and (later) push the LoRA adapter.

**On Kaggle**: Notebook → Add-ons → Secrets → add `HF_TOKEN`. The cell below picks it up automatically.
**On Colab**: Sidebar → key icon → add `HF_TOKEN` to Colab Secrets.
**Locally**: `export HF_TOKEN=...` before launching the kernel.

The cell branches on the runtime: Kaggle reads via `UserSecretsClient`, Colab via `google.colab.userdata`, local just inherits the env var. After the token is in `os.environ`, `huggingface_hub.login()` registers it for every later HF call (datasets, model pull, hub push) without you having to thread the token through each one.

`add_to_git_credential=False` keeps the token out of `~/.gitconfig` — important on shared runtimes where the next session might not be yours.

The `assert` makes the missing-token failure mode loud: a one-line halt instead of an obscure 401 deep inside `load_dataset` later.

### What the output tells you

```
OK: HF authenticated
```

Anything else (a Python traceback, a 401, a "token revoked" warning) means the token is wrong or expired — fix it before continuing or every later HF call will fail.

In [ ]:
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
elif IS_COLAB:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

assert 'HF_TOKEN' in os.environ and os.environ['HF_TOKEN'], 'HF_TOKEN missing'

from huggingface_hub import login as hf_login
hf_login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('OK: HF authenticated')

## 5. Load dataset

The dataset is already published at `Sizzing/aws-rl-sft`. Each row has `messages` (chat format) plus metadata columns (`task_id`, `difficulty`, `source`, `step_idx`) for filtering.

`load_dataset(...)` pulls three parquet shards from the Hub: `train`, `validation`, and a held-out `reserve` split (used later for GRPO-time evaluation, not here). After the call returns, `ds` is a `DatasetDict` whose splits are lazily memory-mapped — no full load until you index into them.

### What the output tells you

```
DatasetDict({
    train:      Dataset({ features: [task_id, difficulty, source, step_idx, messages], num_rows: 1500 })
    validation: Dataset({ features: [task_id, difficulty, source, step_idx, messages], num_rows:  150 })
    reserve:    Dataset({ features: [task_id, difficulty, source, step_idx, messages], num_rows:  200 })
})
```

The sanity-print walks the `messages` list of one training row. Each row has three turns:

- **`system`** — the AWS-engineer persona prompt (constant across the dataset)
- **`user`** — the task description plus environment context (`Step:`, `Last command output:`, …)
- **`assistant`** — the canonical AWS CLI command (this is the supervision target)

Example output:

```
task_id=11  difficulty=intermediate  source=verification
  [system   ] You are an AWS cloud engineer interacting with a real AWS environment via CLI...
  [user     ] TASK: Create an S3 bucket named 'data-pipeline' and upload a file to it...
  [assistant] aws s3api list-objects-v2 --bucket data-pipeline
```

Metadata fields (`difficulty`, `source`, `step_idx`) drive the eval-set curation in §6 — we want coverage across difficulty tiers and source categories, not just easy prompts.

In [ ]:
from datasets import load_dataset

ds = load_dataset(CONFIG['dataset_repo'], token=os.environ['HF_TOKEN'])
print(ds)

# Sanity: look at one row
row = ds['train'][0]
print(f'\ntask_id={row["task_id"]}  difficulty={row["difficulty"]}  source={row["source"]}')
for m in row['messages']:
    preview = m['content'][:140].replace('\n', ' ')
    print(f'  [{m["role"]:<9}] {preview}')

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/194k [00:00<?, ?B/s]

data/reserve-00000-of-00001.parquet:   0%|          | 0.00/261k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/150 [00:00<?, ? examples/s]

Generating reserve split:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['task_id', 'difficulty', 'source', 'step_idx', 'messages'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['task_id', 'difficulty', 'source', 'step_idx', 'messages'],
        num_rows: 150
    })
    reserve: Dataset({
        features: ['task_id', 'difficulty', 'source', 'step_idx', 'messages'],
        num_rows: 200
    })
})

task_id=11  difficulty=intermediate  source=verification
  [system   ] You are an AWS cloud engineer interacting with a real AWS environment via CLI. Each turn you must send exactly ONE valid AWS CLI command (st
  [user     ] TASK: Create an S3 bucket named 'data-pipeline' and upload a file to it.          Step: 2         Last command output: '{"ETag":"\\"d41d8cd9
  [assistant] aws s3api list-objects-v2 --bucket data-pipeline


## 6. Eval harness — define

We'll use this function twice: once on the base model (baseline) and once after training (delta). Same prompts, same scoring — direct before/after comparison.

**Metrics** (match the ones from the model-selection eval):

- `format%` — output starts with `aws ` (no preamble)
- `format_after_extract%` — starts with `aws ` after stripping fences/prose
- `exact%` — extracted command matches canonical exactly
- `service%` — same AWS service (e.g., both `aws s3api`)
- `operation%` — same service AND operation (e.g., both `aws s3api create-bucket`)

### How the harness works

The cell defines four pieces, applied in this order at eval time:

1. **`extract_command(raw)`** — strips ```code fences``` and prose, then returns the first line starting with `aws `. A model that wraps its answer in markdown still scores under `format_after_extract%`.
2. **`score_row(completion, expected)`** — produces a per-row dict with five booleans. The hierarchy is intentional: `format` ⊂ `format_after_extract`, then `service` ⊂ `operation` ⊂ `exact`. A hit on `exact` implies hits on `service` and `operation`.
3. **`curate_eval_set(dataset, max_per_combo=2)`** — keeps at most 2 prompts per `(difficulty, source)` combo so a single tier can't dominate the score. Diverse > large for an 18-prompt eval.
4. **`eval_model(model, tokenizer, eval_set)`** — runs greedy decoding (`do_sample=False, temperature=0.0`) on each prompt, decodes only the *new* tokens (slicing past `inputs.input_ids.shape[1]`), scores the completion, and returns aggregated percentages plus a `_per_row` list (used in §13 for qualitative inspection).

Greedy decoding is deliberate: it makes the eval *deterministic* across runs. Same model + same prompt → same number every time, so the before/after delta is purely about the weights, not sampling noise.

### What the output tells you

```
Eval set: 18 prompts across 9 (tier, source) combos
```

That's `max_per_combo=2 × 9 combos = 18` prompts spanning the difficulty tiers (easy/intermediate/hard) and source categories present in the validation split.

In [ ]:
def extract_command(raw: str) -> str:
    """Strip fences/prose to find the first 'aws ...' line."""
    text = raw.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(l for l in lines if not l.startswith('```')).strip()
    for line in text.split('\n'):
        line = line.strip()
        if line.startswith('aws '):
            return line
    return text

def score_row(completion: str, expected: str) -> dict:
    extracted = extract_command(completion)
    e_tokens = extracted.split()
    exp_tokens = expected.split()
    return {
        'format_ok': completion.strip().startswith('aws '),
        'format_after_extract': extracted.startswith('aws '),
        'exact': extracted == expected.strip(),
        'service': (len(e_tokens) >= 2 and len(exp_tokens) >= 2 and e_tokens[1:2] == exp_tokens[1:2]),
        'operation': (len(e_tokens) >= 3 and len(exp_tokens) >= 3 and e_tokens[2:3] == exp_tokens[2:3]),
    }

def curate_eval_set(dataset, max_per_combo: int = 2):
    seen = {}
    picks = []
    for r in dataset:
        key = (r['difficulty'], r['source'])
        seen[key] = seen.get(key, 0) + 1
        if seen[key] <= max_per_combo:
            picks.append(r)
    return picks

def eval_model(model, tokenizer, eval_set, max_new_tokens: int = 120) -> dict:
    results = []
    model.eval()
    for row in eval_set:
        msgs = row['messages'][:2]  # system + user only
        expected = row['messages'][2]['content']
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        t0 = time.time()
        with torch.inference_mode():
            out_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        dt = time.time() - t0
        completion = tokenizer.decode(out_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        s = score_row(completion, expected)
        s.update({'latency': dt, 'len': len(completion), 'completion': completion, 'expected': expected})
        results.append(s)
    n = len(results)
    return {
        'format_pct':               sum(r['format_ok'] for r in results) / n,
        'format_after_extract_pct': sum(r['format_after_extract'] for r in results) / n,
        'exact_pct':                sum(r['exact'] for r in results) / n,
        'service_pct':              sum(r['service'] for r in results) / n,
        'operation_pct':            sum(r['operation'] for r in results) / n,
        'avg_latency':              sum(r['latency'] for r in results) / n,
        'avg_len':                  sum(r['len'] for r in results) / n,
        '_per_row':                 results,
    }

EVAL_SET = curate_eval_set(ds['validation'], max_per_combo=2)
combos = len(set((r['difficulty'], r['source']) for r in EVAL_SET))
print(f'Eval set: {len(EVAL_SET)} prompts across {combos} (tier, source) combos')

Eval set: 18 prompts across 9 (tier, source) combos


## 7. Pre-SFT baseline eval

Load Qwen2.5-Coder-3B in 4-bit and run the eval set. This is our **"before"** snapshot — we'll compare against it after training.

`FastLanguageModel.from_pretrained` pulls the 4-bit pre-quantised checkpoint (~2 GB on disk vs. 6 GB for fp16). `for_inference()` then swaps in Unsloth's fused-attention kernels — generation gets ~2× faster, no accuracy loss.

After the eval finishes, the cell:

1. Prints the metrics (each as a percentage, plus latency/length averages)
2. Dumps the metric dict (minus `_per_row` so the JSON stays small) to `OUT_DIR/baseline_metrics.json` — survives session restarts so you can rebuild §12's comparison without rerunning §7
3. Frees the base model from VRAM (`del` + `gc.collect` + `torch.cuda.empty_cache`) so Optuna can reload from scratch each trial without OOMing

### What the output tells you

```
=== PRE-SFT BASELINE ===
  format_pct                      33.3%
  format_after_extract_pct       100.0%
  exact_pct                       38.9%
  service_pct                     77.8%
  operation_pct                   61.1%
  avg_latency                     1.60
  avg_len                        85.83
```

### How to read these numbers

- **`format_pct = 33.3%`** — only ⅓ of completions start with `aws ` directly. The base model often pads with prose before the command.
- **`format_after_extract_pct = 100%`** — but every completion *contains* an `aws …` line somewhere. The model knows the shape, it just doesn't lead with it. SFT will cure this — the easiest delta.
- **`exact_pct = 38.9%`** — the hardest target. Exact-match means the same flags, in the same order, with the same values. Real lift here is the headline number.
- **`service_pct (77.8%) > operation_pct (61.1%) > exact_pct (38.9%)`** — the funnel: it picks the right service most of the time, the right operation often, the exact command less often. Each step down the hierarchy reveals where the base model leaks.
- **`avg_latency ≈ 1.6 s/prompt`** — useful as a regression check; SFT shouldn't make generation slower.
- **`avg_len ≈ 86 chars`** — relatively long because the base model wraps the command in prose; expect this to *shrink* after SFT.

Remember these — §12 prints the same table after training, and the deltas tell you what SFT actually bought.

In [ ]:
from unsloth import FastLanguageModel

# Load base model (will cache on disk after first run)
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['base_model'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
    dtype=None,  # auto-select based on GPU
)
FastLanguageModel.for_inference(base_model)  # 2× faster generation via Unsloth kernels

print('Running pre-SFT baseline eval...')
baseline_metrics = eval_model(base_model, base_tokenizer, EVAL_SET)

print('\n=== PRE-SFT BASELINE ===')
for k, v in baseline_metrics.items():
    if k == '_per_row':
        continue
    if 'pct' in k:
        print(f'  {k:<30} {100*v:5.1f}%')
    else:
        print(f'  {k:<30} {v:.2f}')

# Save baseline to disk so we can compare later even if session restarts
with open(OUT_DIR / 'baseline_metrics.json', 'w') as f:
    json.dump({k: v for k, v in baseline_metrics.items() if k != '_per_row'}, f, indent=2)

# Free memory before Optuna starts reloading models per trial
del base_model, base_tokenizer
gc.collect(); torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Running pre-SFT baseline eval...

=== PRE-SFT BASELINE ===
  format_pct                      33.3%
  format_after_extract_pct       100.0%
  exact_pct                       38.9%
  service_pct                     77.8%
  operation_pct                   61.1%
  avg_latency                    1.60
  avg_len                        85.83


## 8. Optuna — objective function

Each trial: load fresh base model → apply LoRA with trial's suggested params → train on 500-row subset for 1 epoch → return eval loss.

**Search space** (5 tunable knobs):

| Param | Range | Intuition |
|---|---|---|
| `lora_r` | {8, 16, 32} | adapter capacity (more rank = more parameters, more flexibility, more risk of overfit) |
| `lora_alpha_mul` | {1, 2, 4} | effective α = r × mul; α/r is the LoRA scaling factor |
| `lora_dropout` | [0.0, 0.1] | regularisation on the LoRA path |
| `learning_rate` | [1e-4, 5e-4] log | the biggest lever — sampled log-uniform |
| `warmup_ratio` | {0.03, 0.1} | stability at the start of training |

### What the cell does, top to bottom

1. **Transformers 5.x compatibility shim** — Unsloth 2026.x still calls `Trainer(tokenizer=…)`, but Transformers 5.0 renamed it to `processing_class=`. The shim wraps Unsloth's `_original_trainer_init` so the rename happens at the boundary; it's idempotent (guarded by the `_unsloth_tokenizer_shim` flag) so re-running the cell is safe. On Transformers 4.x the shim is skipped.
2. **Trial slices prepared once** — `TRIAL_TRAIN` is a 500-row shuffled subset, `TRIAL_VAL` is 80 rows. They're computed *outside* `objective()` so the random shuffle is constant across trials — every trial sees the same data, only the hparams change.
3. **`_render_messages`** — applies the tokenizer's chat template (Qwen2.5 ChatML) to flatten `messages` → a single `text` string. TRL 0.12+ requires an explicit text column.
4. **`objective(trial)`** is what Optuna calls per trial:
   - Suggests 5 hparams. `lora_alpha = lora_r × lora_alpha_mul` ensures α scales with r as the LoRA paper recommends.
   - Loads a *fresh* base model — same weights, same 4-bit quant, no leakage between trials.
   - `get_peft_model` wraps it with the trial's LoRA config. `use_gradient_checkpointing='unsloth'` recomputes activations to fit T4 VRAM.
   - Pre-tokenises both splits down to `input_ids` only. **Why pre-tokenise?** Unsloth pins `remove_unused_columns=False`; if the `text` column survives into the collator, the trainer raises `too many dimensions 'str'`. Pre-tokenising + `skip_prepare_dataset=True` sidesteps both issues.
   - Builds an `SFTTrainer` + `SFTConfig`. Note `save_strategy='no'` (trials don't checkpoint), `eval_strategy='epoch'` (single eval at end), `report_to='none'` (no wandb).
   - **`train_on_responses_only`** — masks the loss for everything before `<|im_start|>assistant\n`. The model only learns to produce the assistant turn, not memorise the system prompt or user task. This *completion-only loss* is essential — without it the trainer would also try to predict the system prompt and your loss numbers would be meaningless.
   - Trains, evaluates, stashes `log_history` on the trial via `set_user_attr` (used later by `plot_optuna_trial_loss_curves`), prints a one-line summary, frees memory, and returns `eval_loss` for Optuna to minimise.

### What the output tells you

This cell is just defining the function — no training output yet. Expect a single line:

```
Trial train: 500, trial val: 80
```

confirming the slices are sized correctly. Plus, on Transformers 4.x: `Transformers 4.57.6: no shim needed`. On 5.x: `Applied Transformers 5.x.y shim: tokenizer= -> processing_class=`. The actual per-trial logging happens in §9.

In [ ]:
# --- Transformers 5.x compatibility shim (no restart needed) ---
# Unsloth 2026.x forwards `tokenizer=` through **kwargs to Trainer.__init__.
# Transformers 5.0+ removed that kwarg in favor of `processing_class=`.
# We patch Unsloth's captured reference so the rename happens at the boundary.
try:
    import transformers as _tf
    import unsloth.models._utils as _u
    _major = int(_tf.__version__.split('.')[0])
    if _major >= 5 and not getattr(_u, '_unsloth_tokenizer_shim', False):
        _orig_init = _u._original_trainer_init
        def _shimmed_init(self, *args, **kwargs):
            if 'tokenizer' in kwargs:
                kwargs['processing_class'] = kwargs.pop('tokenizer')
            return _orig_init(self, *args, **kwargs)
        _u._original_trainer_init = _shimmed_init
        _u._unsloth_tokenizer_shim = True
        print(f'Applied Transformers {_tf.__version__} shim: tokenizer= -> processing_class=')
    else:
        print(f'Transformers {_tf.__version__}: no shim needed')
except Exception as _e:
    print(f'Shim skipped: {_e}')

import optuna
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Prepare trial-sized splits once (not per trial)
TRIAL_TRAIN = ds['train'].shuffle(seed=CONFIG['seed']).select(range(CONFIG['trial_max_train_rows']))
TRIAL_VAL = ds['validation'].select(range(min(80, len(ds['validation']))))
print(f'Trial train: {len(TRIAL_TRAIN)}, trial val: {len(TRIAL_VAL)}')

def _render_messages(example, tokenizer):
    """Apply the chat template to turn messages into a single text string.
    TRL 0.12+ requires an explicit text column; Unsloth respects the
    tokenizer's built-in Qwen2.5 ChatML template.
    """
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False,
    )}

def objective(trial: optuna.Trial) -> float:
    """One Optuna trial. Returns eval_loss (minimize)."""
    gc.collect(); torch.cuda.empty_cache()

    # --- Suggest hyperparameters ---
    lora_r          = trial.suggest_categorical('lora_r', [8, 16, 32])
    lora_alpha_mul  = trial.suggest_categorical('lora_alpha_mul', [1, 2, 4])
    lora_alpha      = lora_r * lora_alpha_mul
    lora_dropout    = trial.suggest_float('lora_dropout', 0.0, 0.1)
    learning_rate   = trial.suggest_float('learning_rate', 1e-4, 5e-4, log=True)
    warmup_ratio    = trial.suggest_categorical('warmup_ratio', [0.03, 0.1])

    # --- Load fresh model + tokenizer for this trial ---
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG['base_model'],
        max_seq_length=CONFIG['max_seq_length'],
        load_in_4bit=CONFIG['load_in_4bit'],
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=CONFIG['lora_target_modules'],
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=CONFIG['seed'],
    )

    # --- Render messages -> text using THIS trial's tokenizer ---
    train_ds = TRIAL_TRAIN.map(lambda ex: _render_messages(ex, tokenizer))
    val_ds   = TRIAL_VAL.map(lambda ex: _render_messages(ex, tokenizer))
    # Drop metadata and pre-tokenize, leaving only numeric columns.
    # Unsloth forces remove_unused_columns=False; if `text` survives into
    # the collator it raises 'too many dimensions str'. Pre-tokenizing and
    # skipping the trainer's internal dataset prep sidesteps both issues.
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c != 'text'])
    val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c != 'text'])
    def _tokenize(batch):
        return tokenizer(
            batch['text'], truncation=True,
            max_length=CONFIG['max_seq_length'], padding=False,
        )
    train_ds = train_ds.map(_tokenize, batched=True, remove_columns=['text'])
    val_ds   = val_ds.map(_tokenize, batched=True, remove_columns=['text'])

    trial_dir = OUT_DIR / f'optuna/trial-{trial.number}'
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=SFTConfig(
            output_dir=str(trial_dir),
            dataset_text_field='text',
            dataset_kwargs={'skip_prepare_dataset': True},  # we pre-tokenized above
            max_seq_length=CONFIG['max_seq_length'],
            per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
            gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
            num_train_epochs=CONFIG['trial_epochs'],
            learning_rate=learning_rate,
            warmup_ratio=warmup_ratio,
            lr_scheduler_type=CONFIG['lr_scheduler_type'],
            optim=CONFIG['optim'],
            weight_decay=CONFIG['weight_decay'],
            max_grad_norm=CONFIG['max_grad_norm'],
            fp16=USE_FP16, bf16=USE_BF16,
            logging_steps=25,
            eval_strategy='epoch',
            save_strategy='no',                    # trials don't save
            report_to='none',
            run_name=f'optuna-trial-{trial.number}',
            seed=CONFIG['seed'],
            dataset_num_proc=1,
        ),
    )
    # Completion-only loss: mask system + user tokens; train only on the
    # assistant's reply. These ChatML delimiters are Qwen2.5-specific.
    trainer = train_on_responses_only(
        trainer,
        instruction_part='<|im_start|>user\n',
        response_part='<|im_start|>assistant\n',
    )

    trainer.train()
    eval_result = trainer.evaluate()

    # Stash the trainer's log_history on the trial so plot_optuna_trial_loss_curves
    # can overlay every trial's curves later without a custom TrainerCallback.
    trial.set_user_attr('log_history', list(trainer.state.log_history))

    loss = eval_result['eval_loss']
    print(f'  Trial {trial.number}: r={lora_r} alpha={lora_alpha} dropout={lora_dropout:.3f} '
          f'lr={learning_rate:.2e} warmup={warmup_ratio} -> eval_loss={loss:.4f}')

    del model, tokenizer, trainer, train_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()
    return loss

## 9. Run the Optuna study

TPE sampler (Bayesian-ish) is more sample-efficient than random with only 6 trials.

`optuna.create_study(direction='minimize')` makes a new in-memory study. `TPESampler(seed=42)` is deterministic — same seed, same trial sequence, so the search is reproducible. With only 6 trials a Tree-structured Parzen Estimator typically beats random because it builds a probability density of "good" vs "bad" hparams after the first few trials and biases later suggestions toward the good region.

`study.optimize(objective, n_trials=6)` runs the loop. Each trial logs:

- The Optuna `[I ...]` info line with parameters and the value
- The notebook's own one-line summary (`Trial N: r=… alpha=… → eval_loss=…`)
- An Unsloth banner showing trainable params (e.g. `7,372,800 of 3,093,311,488 (0.24% trained)` for r=16)

After the loop, `study.best_value` / `study.best_params` are populated. The cell pretty-prints them and dumps the full study (every trial's params and value) to `OUT_DIR/optuna_study.json` so plots can be rebuilt without rerunning.

### What the output tells you

The 6 per-trial summaries from this run, followed by the winner:

```
Trial 0: r=16 alpha=16 dropout=0.006 lr=4.03e-04 warmup=0.1   -> eval_loss=0.0523
Trial 1: r=16 alpha=16 dropout=0.030 lr=2.33e-04 warmup=0.03  -> eval_loss=0.0790
Trial 2: r=8  alpha=32 dropout=0.020 lr=2.29e-04 warmup=0.03  -> eval_loss=0.0587
Trial 3: r=8  alpha=16 dropout=0.030 lr=1.17e-04 warmup=0.03  -> eval_loss=0.1199
Trial 4: r=16 alpha=16 dropout=0.031 lr=2.31e-04 warmup=0.03  -> eval_loss=0.0793
Trial 5: r=8  alpha=32 dropout=0.009 lr=1.37e-04 warmup=0.1   -> eval_loss=0.0828

=== OPTUNA RESULTS ===
Best eval_loss : 0.0523
Best trial     : #0
Best params    :
  lora_r               16
  lora_alpha_mul       1
  lora_dropout         0.0058
  learning_rate        4.03e-04
  warmup_ratio         0.1
```

Trial #0 won — TPE got lucky on its first sample, then the next five didn't beat it. That's normal for a small-budget search; the spread between best (0.0523) and worst (0.1199) trials is ~2× in eval loss, which is exactly the kind of separation §10's plots will visualise. Note also that the lowest-LR trial (#3, lr=1.17e-04) had the worst eval loss — strong evidence that learning rate is the dominant knob in this range.

In [ ]:
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=CONFIG['seed']),
    study_name='aws-rl-sft-lora-search',
)
study.optimize(objective, n_trials=CONFIG['n_trials'])

print('\n=== OPTUNA RESULTS ===')
print(f'Best eval_loss : {study.best_value:.4f}')
print(f'Best trial     : #{study.best_trial.number}')
print(f'Best params    :')
for k, v in study.best_params.items():
    print(f'  {k:<20} {v}')

# Persist the study so we can replot without rerunning
with open(OUT_DIR / 'optuna_study.json', 'w') as f:
    json.dump({
        'best_value': study.best_value,
        'best_params': study.best_params,
        'trials': [{'number': t.number, 'value': t.value, 'params': t.params, 'state': str(t.state)}
                   for t in study.trials],
    }, f, indent=2)

[I 2026-04-22 23:58:32,139] A new study created in memory with name: aws-rl-sft-lora-search


==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.005808361216819946.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 32
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 3,686,400 of 3,089,625,088 (0.12% trained)


Epoch,Training Loss,Validation Loss
1,0.122500,0.082837


eval/loss,█▁
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
train/epoch,▁███
train/global_step,▁███
train/grad_norm,▁
train/learning_rate,▁
train/loss,▁
eval/loss,0.08283
eval/runtime,13.5748


[I 2026-04-23 00:21:57,943] Trial 5 finished with value: 0.08283159881830215 and parameters: {'lora_r': 8, 'lora_alpha_mul': 4, 'lora_dropout': 0.00884925020519195, 'learning_rate': 0.0001370838023704289, 'warmup_ratio': 0.1}. Best is trial 0 with value: 0.05230738967657089.


  Trial 5: r=8 alpha=32 dropout=0.009 lr=1.37e-04 warmup=0.1 -> eval_loss=0.0828

=== OPTUNA RESULTS ===
Best eval_loss : 0.0523
Best trial     : #0
Best params    :
  lora_r               16
  lora_alpha_mul       1
  lora_dropout         0.005808361216819946
  learning_rate        0.00040311702880369243
  warmup_ratio         0.1


## 10. Optuna plots

Five views, all rendered with matplotlib and saved as PNGs to `OUT_DIR/plots/`:

1. **Optimization history** — did later trials converge on lower loss?
2. **Param importances** — which knob mattered most (fANOVA-style)?
3. **Parallel coordinates** — which hparam combinations trend toward low loss?
4. **Slice plot** — per-parameter scatter against the objective.
5. **Per-trial loss curves** — overlay every trial's training/eval loss to see how the search space behaved during fitting.

### How to read each plot

- **`optuna_history.png`** — blue dots are individual trial values; the red staircase is the best-so-far. A flat staircase past trial 1 means TPE didn't find anything better than the first random pick (expected with only 6 trials, and what happens in this run since trial #0 won outright).
- **`optuna_importances.png`** — fANOVA-style: how much variance in `eval_loss` each parameter explains. Given the trial table from §9, expect `learning_rate` to dominate; if `lora_dropout` shows up high it means you under-regularised the larger ranks.
- **`optuna_parallel.png`** — every trial is one polyline across all hparam axes (last axis = objective). Polyline colour encodes the objective value (viridis_r — dark = lower / better). Look for clusters of dark lines over the same axis values — that's the winning region.
- **`optuna_slice.png`** — one panel per param: x-axis is the param value, y-axis is `eval_loss`, dot colour encodes trial number. A clear monotonic trend means the search space was pointed in the right direction; a flat scatter means the param was insensitive in this range.
- **`optuna_trial_curves.png`** — every trial's training-loss + eval-loss curves overlaid on the same axes. The trial that lands lowest at the end is your best — but watch for trials that *would have* dropped further with more steps (early-stopped winners). With only 32 steps per trial these curves are short, so use them to confirm trends, not for fine analysis.

### What the output tells you

Each helper prints a `saved -> /path/to/png` line as it writes its figure to disk and shows the chart inline. Five PNGs land in `OUT_DIR/plots/` (`optuna_history.png`, `optuna_importances.png`, `optuna_parallel.png`, `optuna_slice.png`, `optuna_trial_curves.png`) — all together they tell the story of *why* the final hparams were chosen.

In [ ]:
# All Optuna views as matplotlib PNGs in OUT_DIR/plots/.
# Each helper renders inline AND saves to disk; see PLOTS_DIR for the files.
plot_optuna_history(study)
plot_optuna_param_importances(study)
plot_optuna_parallel_coordinates(study)
plot_optuna_slice(study)
plot_optuna_trial_loss_curves(study)

## 11. Final SFT run — best hparams, full data, checkpointed

- Full dataset (1500 train / 150 val)
- 2 epochs (vs. 1 during search)
- Checkpoints saved every 50 steps; last 3 kept (`save_total_limit=3`)
- `load_best_model_at_end=True` — final model = lowest eval_loss checkpoint, not last
- **Resume-safe**: if the session dies, rerunning this cell picks up from the latest `checkpoint-*/`

### What this cell does

The structure mirrors `objective()` but with three meaningful differences: full data, more epochs, and checkpointing.

1. **Re-apply the Transformers 5.x shim** (idempotent — same flag-guard as §8).
2. **Reload base + apply LoRA** with the best hparams from `study.best_params`. `final_alpha = final_r × best['lora_alpha_mul']`.
3. **Render and pre-tokenise the full dataset** (1500 train / 150 val) — same chat-template + drop-metadata + tokenize trick as the trial loop, just on more data.
4. **Auto-detect resume checkpoint** — globs `OUT_DIR/final_sft/checkpoint-*` and picks the highest step number. If found, prints `Resuming from checkpoint: …` and `trainer.train(resume_from_checkpoint=…)` continues from there. If the session dies mid-training (T4 timeouts on free Colab/Kaggle), just rerun this cell — no edits needed.
5. **Build the final `SFTTrainer`** with `load_best_model_at_end=True` + `metric_for_best_model='eval_loss'` + `greater_is_better=False`. After training finishes, the trainer reloads the checkpoint with the lowest validation loss — so your final weights are *not* necessarily the last step's weights. This is your built-in protection against late-training overfit.
6. **`train_on_responses_only`** — same completion-only loss masking. Critical for fair comparison with the Optuna trials (apples-to-apples loss values).
7. **Evaluate once at the end** and print both the average training loss and the final eval loss.

### What the output tells you

Total steps ≈ 1500 rows × 2 epochs ÷ effective-batch 16 ≈ 187 steps. Along the way you'll see Unsloth's banner reporting `Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)` (assuming the winning `r=16`), then a stream of progress bars from the trainer logging every 10 steps and evaluating every 50.

The final two lines summarise:

```
Final train loss: ~0.04
Final eval loss:  ~0.05
```

A healthy run has training loss dropping smoothly with no spikes, eval loss tracking close to training (small overfit gap), and the LR cosine-annealing to ~0 by the end. If `eval_loss` is rising in the last few evals while `loss` is still falling, that's overfitting — `load_best_model_at_end` will rescue you by reloading an earlier checkpoint.

In [ ]:
# Ensure the Transformers 5.x shim is in place (idempotent).
try:
    import transformers as _tf
    import unsloth.models._utils as _u
    if int(_tf.__version__.split('.')[0]) >= 5 and not getattr(_u, '_unsloth_tokenizer_shim', False):
        _orig_init = _u._original_trainer_init
        def _shimmed_init(self, *args, **kwargs):
            if 'tokenizer' in kwargs:
                kwargs['processing_class'] = kwargs.pop('tokenizer')
            return _orig_init(self, *args, **kwargs)
        _u._original_trainer_init = _shimmed_init
        _u._unsloth_tokenizer_shim = True
except Exception:
    pass

gc.collect(); torch.cuda.empty_cache()

best = study.best_params
final_r = best['lora_r']
final_alpha = final_r * best['lora_alpha_mul']

# --- Fresh model with winning hparams ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['base_model'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=final_r,
    lora_alpha=final_alpha,
    lora_dropout=best['lora_dropout'],
    target_modules=CONFIG['lora_target_modules'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=CONFIG['seed'],
)

# --- Render the full dataset with the chat template ---
full_train = ds['train'].map(lambda ex: _render_messages(ex, tokenizer))
full_val   = ds['validation'].map(lambda ex: _render_messages(ex, tokenizer))
# Drop metadata then pre-tokenize — leaves only numeric columns for the collator.
full_train = full_train.remove_columns([c for c in full_train.column_names if c != 'text'])
full_val   = full_val.remove_columns([c for c in full_val.column_names if c != 'text'])
def _tokenize_batch(batch):
    return tokenizer(
        batch['text'], truncation=True,
        max_length=CONFIG['max_seq_length'], padding=False,
    )
full_train = full_train.map(_tokenize_batch, batched=True, remove_columns=['text'])
full_val   = full_val.map(_tokenize_batch, batched=True, remove_columns=['text'])

# --- Auto-detect existing checkpoint to resume ---
ckpt_root = OUT_DIR / 'final_sft'
resume_ckpt = None
if ckpt_root.exists():
    existing = sorted(
        [d for d in ckpt_root.glob('checkpoint-*') if d.is_dir()],
        key=lambda d: int(d.name.split('-')[-1]),
    )
    if existing:
        resume_ckpt = str(existing[-1])
        print(f'Resuming from checkpoint: {resume_ckpt}')
    else:
        print('No checkpoint found — starting fresh')
else:
    print('No prior training dir — starting fresh')

final_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=full_train,
    eval_dataset=full_val,
    args=SFTConfig(
        output_dir=str(ckpt_root),
        dataset_text_field='text',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_seq_length=CONFIG['max_seq_length'],
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        num_train_epochs=CONFIG['num_train_epochs'],
        learning_rate=best['learning_rate'],
        warmup_ratio=best['warmup_ratio'],
        lr_scheduler_type=CONFIG['lr_scheduler_type'],
        optim=CONFIG['optim'],
        weight_decay=CONFIG['weight_decay'],
        max_grad_norm=CONFIG['max_grad_norm'],
        fp16=USE_FP16, bf16=USE_BF16,
        eval_strategy='steps',
        eval_steps=CONFIG['eval_steps'],
        save_strategy='steps',
        save_steps=CONFIG['save_steps'],
        save_total_limit=CONFIG['save_total_limit'],
        logging_steps=CONFIG['logging_steps'],
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        run_name='final-sft',
        seed=CONFIG['seed'],
        dataset_num_proc=1,
    ),
)
# Completion-only loss — same masking as Optuna trials
final_trainer = train_on_responses_only(
    final_trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

train_result = final_trainer.train(resume_from_checkpoint=resume_ckpt)
print(f'\nFinal train loss: {train_result.training_loss:.4f}')
final_eval = final_trainer.evaluate()
print(f'Final eval loss:  {final_eval["eval_loss"]:.4f}')

## 11a. Final SFT — training curves

The trainer's `log_history` carries every logged step (loss, eval_loss, learning_rate, grad_norm). Render it as a 2×2 grid and save to `OUT_DIR/plots/final_training_curves.png`.

`final_trainer.state.log_history` is a list of dicts — one per logging event. Three event types live there:

- **training rows**: `{'step': N, 'loss': …, 'learning_rate': …, 'grad_norm': …}` (every `logging_steps=10`)
- **eval rows**: `{'step': N, 'eval_loss': …}` (every `eval_steps=50`)
- a final summary row with neither `loss` nor `eval_loss` (silently dropped by `_split_log_history`)

`plot_training_curves` produces a 2×2 figure:

| Panel | What it shows | What "good" looks like |
|---|---|---|
| **Top-left** — training loss | Cross-entropy per step on the masked assistant tokens | Monotonically decreasing curve, no spikes; some noise from gradient accumulation is normal |
| **Top-right** — eval loss | Cross-entropy on the held-out val set every 50 steps | Tracks training loss; final value close to (or below) the best Optuna trial's eval loss (~0.05 for this run) |
| **Bottom-left** — learning rate | The cosine schedule with warmup | Linear ramp up for the first ~3-10% of steps (per the chosen `warmup_ratio`), then a smooth cosine decay to ~0 by the last step |
| **Bottom-right** — gradient norm | ‖grad‖₂ pre-clip | Settles into a stable range (often 0.5–2.0); spikes followed by NaN losses mean lower the LR |

### What the output tells you

The cell renders the figure inline and writes `OUT_DIR/plots/final_training_curves.png` with a `saved -> …` confirmation line. Together with `OUT_DIR/plots/eval_metrics_comparison.png` from §12, this is the figure most likely to make it into the write-up — it's the proof the run was healthy.

In [ ]:
# Final SFT — training curves
# (train + eval loss, learning-rate schedule, gradient-norm) extracted from
# trainer.state.log_history. Saved to OUT_DIR/plots/final_training_curves.png.
plot_training_curves(
    list(final_trainer.state.log_history),
    title='Final SFT — training curves (full dataset, best Optuna params)',
    save_name='final_training_curves',
)

## 12. Post-SFT eval — the delta that matters

Same prompts as the baseline. This is the headline table for judges.

The cell switches the model into Unsloth's fast-inference mode (`for_inference`) and runs `eval_model` on the *same 18-prompt `EVAL_SET`* used for the baseline — identical prompts, identical scoring, only the weights have changed. Then it prints a side-by-side table, dumps the numbers to JSON, and renders the comparison plot.

### What the output tells you

A formatted Before / After / Delta table for every metric. The shape of the change (with the baseline numbers from §7 as anchors):

| Metric | Before | Direction expected after SFT | Why |
|---|--:|---|---|
| `format_pct` | 33.3% | jumps to near 100% | Largest absolute delta — the base model knew the shape but buried it in prose; SFT teaches it to lead with `aws `. |
| `format_after_extract_pct` | 100.0% | stays at 100% | No headroom; an identity result is the right outcome. |
| `exact_pct` | 38.9% | ~doubles | The headline number — same flags, same order, same values. §13's qualitative samples (5 of 6 exact matches in the first slice) imply this lands in the 80%+ range. |
| `service_pct` | 77.8% | rises further | Already partly correct in the baseline; ceiling is the eval-set's task variety. |
| `operation_pct` | 61.1% | rises substantially | Falls between `service` and `exact` in the funnel. |
| `avg_len` | 85.83 | shrinks | Shorter outputs because the model stops emitting prose around the command. |

### Other artefacts

The cell also dumps `OUT_DIR/delta_summary.json` — a `{baseline, posttrain, delta}` dict for the write-up — and `plot_eval_comparison` saves `OUT_DIR/plots/eval_metrics_comparison.png`. That figure has two panels: grouped bars on the left (blue baseline, orange post-SFT) with percentage labels above each bar, and a delta-pp bar chart on the right (green = improvement, red = regression). The delta-pp panel is the single most decision-relevant figure in the notebook.

In [ ]:
FastLanguageModel.for_inference(model)

print('Running post-SFT eval on the same EVAL_SET...')
posttrain_metrics = eval_model(model, tokenizer, EVAL_SET)

print('\n=== BEFORE vs AFTER ===')
print(f'{"Metric":<30} {"Before":>10} {"After":>10} {"Delta":>12}')
print('-' * 64)
metric_keys = ['format_pct', 'format_after_extract_pct', 'exact_pct',
               'service_pct', 'operation_pct', 'avg_latency', 'avg_len']
for k in metric_keys:
    b = baseline_metrics[k]
    a = posttrain_metrics[k]
    if 'pct' in k:
        print(f'{k:<30} {100*b:9.1f}% {100*a:9.1f}% {100*(a-b):+11.1f}pt')
    else:
        print(f'{k:<30} {b:10.2f} {a:10.2f} {a-b:+12.2f}')

# Persist for the write-up
with open(OUT_DIR / 'delta_summary.json', 'w') as f:
    json.dump({
        'baseline': {k: baseline_metrics[k] for k in metric_keys},
        'posttrain': {k: posttrain_metrics[k] for k in metric_keys},
        'delta': {k: posttrain_metrics[k] - baseline_metrics[k] for k in metric_keys},
    }, f, indent=2)

# Render baseline-vs-SFT grouped bars + Δpp delta bars; saved to
# OUT_DIR/plots/eval_metrics_comparison.png. The JSON dump above keeps the
# raw numbers for any downstream comparison.
plot_eval_comparison(baseline_metrics, posttrain_metrics)

## 13. Inspect a few post-SFT generations

Qualitative sanity: does the trained model actually produce valid AWS commands?

The aggregate metrics in §12 can hide failure modes (e.g., the model collapsing to one command). This cell prints six side-by-side comparisons of expected vs generated, prefixed with one of three markers:

- **`[OK]`** — `exact` match (`r['exact'] == True`)
- **`[~ ]`** — formatted correctly but the command differs (`format_after_extract` only)
- **`[X ]`** — couldn't even extract an `aws …` line

### What the output tells you

```
Post-SFT generations vs canonical:

  [OK] expected : 'aws route53 list-hosted-zones'
       generated: 'aws route53 list-hosted-zones'

  [OK] expected : 'aws dynamodb put-item --table-name orders --item ...'
       generated: 'aws dynamodb put-item --table-name orders --item ...'

  [~ ] expected : 'aws help --task-hint'
       generated: 'aws lambda create-function --function-name scheduled-task ...'

  [OK] expected : 'aws sns create-topic --name notifications'
       generated: 'aws sns create-topic --name notifications'

  [OK] expected : 'aws apigatewayv2 create-api --name payments-api --protocol-type HTTP'
       generated: 'aws apigatewayv2 create-api --name payments-api --protocol-type HTTP'

  [OK] expected : 'aws s3api create-bucket --bucket firehose-delivery'
       generated: 'aws s3api create-bucket --bucket firehose-delivery'
```

5 / 6 exact matches in the first slice. The single `[~ ]` is interesting: the canonical answer is the meta-command `aws help --task-hint`, which is a dataset-specific control token, not a real AWS API call. The model produces a real command instead — a *reasonable* error that the next phase (GRPO with environment reward) can correct, since the live environment will signal whether the chosen command actually advanced the task.

Use this section to spot patterns: are failures concentrated in one service? Are flag values being hallucinated? Are JSON args malformed? Each pattern points at a different fix (more data, better masking, a specific reward in GRPO).

In [ ]:
print('Post-SFT generations vs canonical:')
for r in posttrain_metrics['_per_row'][:6]:
    match = 'OK' if r['exact'] else ('~ ' if r['format_after_extract'] else 'X ')
    print(f'\n  [{match}] expected : {r["expected"][:100]!r}')
    print(f'      generated: {r["completion"].strip()[:100]!r}')

Post-SFT generations vs canonical:

  [OK] expected : 'aws route53 list-hosted-zones'
      generated: 'aws route53 list-hosted-zones'

  [OK] expected : 'aws dynamodb put-item --table-name orders --item \'{"order_id":{"S":"001"},"status":{"S":"pending"}}\''
      generated: 'aws dynamodb put-item --table-name orders --item \'{"order_id":{"S":"001"},"status":{"S":"pending"}}\''

  [~ ] expected : 'aws help --task-hint'
      generated: 'aws lambda create-function --function-name scheduled-task --runtime python3.12 --handler index.handl'

  [OK] expected : 'aws sns create-topic --name notifications'
      generated: 'aws sns create-topic --name notifications'

  [OK] expected : 'aws apigatewayv2 create-api --name payments-api --protocol-type HTTP'
      generated: 'aws apigatewayv2 create-api --name payments-api --protocol-type HTTP'

  [OK] expected : 'aws s3api create-bucket --bucket firehose-delivery'
      generated: 'aws s3api create-bucket --bucket firehose-delivery'


## 14. Push LoRA adapter to Hugging Face Hub

Just the adapter — ~60MB instead of the full 3B (~6GB). Consumers will apply it on top of `Qwen/Qwen2.5-Coder-3B-Instruct` at inference time.

`model.save_pretrained` writes the LoRA weights and the PEFT config (`adapter_config.json`, `adapter_model.safetensors`) to disk; `tokenizer.save_pretrained` writes the tokenizer files (`tokenizer.json`, `vocab.json`, `merges.txt`, `special_tokens_map.json`, …). PEFT only persists the adapter matrices (~15 MB at `r=16`); the tokenizer adds ~12 MB. The Hub-side upload deduplicates against existing files, so re-pushes only send what changed.

`push_to_hub(..., private=True)` creates the repo if it doesn't exist and uploads. Private by default so you can iterate before sharing.

### What the output tells you

```
Adapter saved locally: /content/out/adapter
...adapter_model.safetensors:  100%|##########| 14.8MB / 14.8MB
Saved model to https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter
...tokenizer.json:             100%|##########| 11.4MB / 11.4MB
OK: adapter pushed to https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter
```

The first upload pushes the 14.8 MB safetensors blob (the actual LoRA matrices). The second pushes the 11.4 MB `tokenizer.json` plus the smaller config/vocab files. Total payload ~30 MB — orders of magnitude lighter than shipping a fine-tuned 3B model. The two `Saved model to …` / `OK: …` lines confirm both uploads succeeded; the URL goes straight to the adapter's Hub page.

In [ ]:
adapter_local = OUT_DIR / 'adapter'
model.save_pretrained(str(adapter_local))
tokenizer.save_pretrained(str(adapter_local))
print(f'Adapter saved locally: {adapter_local}')

model.push_to_hub(CONFIG['adapter_repo'], private=CONFIG['adapter_private'], token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub(CONFIG['adapter_repo'], private=CONFIG['adapter_private'], token=os.environ['HF_TOKEN'])
print(f'OK: adapter pushed to https://huggingface.co/{CONFIG["adapter_repo"]}')

Adapter saved locally: /content/out/adapter


README.md:   0%|          | 0.00/573 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|3         |  525kB / 14.8MB            

Saved model to https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcnfz5qff/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

OK: adapter pushed to https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter


## 15. How to use this adapter at inference time

Because we pushed only the LoRA adapter, consumers don't need to host the full 3B base — Unsloth's loader handles the merge:

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Sizzing/aws-rl-sft-qwen25coder3b-adapter',  # adapter repo
    max_seq_length=512,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
# Unsloth auto-pulls the base model and attaches the LoRA
```

The adapter's `adapter_config.json` records the base model (`Qwen/Qwen2.5-Coder-3B-Instruct`), so passing the adapter repo to `from_pretrained` is enough — Unsloth resolves the base, downloads it once (cached locally for subsequent runs), and attaches the LoRA on top. `for_inference` then swaps in the fast-attention kernels for ~2× faster generation.

## 16. Next phase — GRPO

With the SFT'd adapter published, the GRPO phase builds on top:

1. Load base model + this adapter as the policy
2. Same adapter, frozen, as the KL reference (ensures GRPO doesn't drift)
3. Roll out episodes against the live `aws-rl-env` (MiniStack)
4. GRPO with online env reward + optional `<think>` reasoning (R1-Zero style)

The adapter we just trained serves as both the starting policy and the KL anchor — SFT locks format and basic competence, GRPO refines task correctness with real reward signal.

### Why SFT first, then GRPO?

GRPO is a policy-gradient method — it needs a starting policy that can already produce well-formatted, plausibly-correct outputs, otherwise the reward signal is too sparse to learn from. The 33% → ~95%+ format jump and the 39% → ~80%+ exact-match jump that this SFT run buys mean GRPO can spend its compute exploring *which command* is best for a task, not learning that commands should start with `aws `.

In [ ]:
import shutil
from pathlib import Path

STAGE = Path('/content/aws_rl_sft_export')
STAGE.mkdir(exist_ok=True)

# Copy the output folder wholesale. `out/plots/` is included automatically.
for folder in ['out']:
    src = Path('/content') / folder
    if src.exists():
        dest = STAGE / folder
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        size_mb = sum(p.stat().st_size for p in dest.rglob('*') if p.is_file()) / 1e6
        print(f'  {folder}/  {size_mb:.1f} MB')
    else:
        print(f'  {folder}/  MISSING')

# Zip it
zip_path = shutil.make_archive('/content/aws_rl_sft_artifacts', 'zip', STAGE)
size_mb = Path(zip_path).stat().st_size / 1e6
print(f'\nArchive: {zip_path}  ({size_mb:.1f} MB)')

# Download
from google.colab import files
files.download(zip_path)